# Suburbia — Scraper de celulares (anti bot-blocker)

**Navegación como en el video:** `home → Categorías → Celulares` (seleccionado por `href`) → filtro **Vendido por: suburbia** → paginar, todo en **una sola sesión** stealth.

**Cambios de esta versión (rediseño del sitio):**
- Celulares se selecciona por su `href` real (`/tienda/celulares/catst53278605`), no por texto.
- **Parser independiente de las clases**: se ancla en los enlaces `/tienda/pdp/` y saca marca/nombre/precio por patrón (las clases viejas como `m-product__card` ya no existen).
- **SKU/URL** en el formato real del PLP `/tienda/pdp/<slug>/<CODIGO>?skuid=<SKU>`. `CodigoProducto` = lo de antes del `?` (ej. `SB5012984724`); `SKU` = el `skuid` (ej. `SB5012984733`).
- Reúne los **centavos en superíndice** (`$4,749 ²⁹` → `$4,749.29`).
- **Ignora la mensualidad** (`Desde $X quincenales`) para que ese pago no se cuele como precio del celular.
- Si una página no trae productos, guarda `suburbia_debug_*.html` para inspección.

El cruce con el diccionario trae **Capacidad y Color** por SKU. Si un SKU tiene **varios colores** en el diccionario (una fila por color), el resultado se expande a **una fila por color** para ese producto. La exportación a Excel es la de tu versión original.

**Instalación (recomendado incluir undetected-chromedriver):**
```
pip install undetected-chromedriver selenium beautifulsoup4 pandas openpyxl
```

**Flags arriba del código:** `RUTA_DICCIONARIO`, `MAX_PAGINAS`, `HEADLESS`, `EXPANDIR_VARIANTES` (entrar a cada PDP; por defecto `False`), `DEBUG_HTML`.

In [5]:
# -*- coding: utf-8 -*-
"""
Suburbia - Scraper de celulares (versión anti-bot-blocker)
===========================================================
Cambio principal vs. la versión original:
  - Antes: se abría DIRECTO la URL codificada de la categoría (/tienda/celulares/N-7%2F...)
    y se lanzaban MUCHOS ChromeDrivers headless en paralelo. Eso es justo lo que un
    bot blocker (Akamai / DataDome / PerimeterX) detecta y bloquea.
  - Ahora: UNA sola sesión de navegador "stealth" que entra como humano:
        home  ->  Categorías  ->  Electrónica  ->  Celulares
        ->  filtro "Vendido por: suburbia"  ->  ir cambiando de página (paginación)
    Las cookies/tokens que pone el sitio durante esa navegación son los que dejan pasar
    las siguientes peticiones (listado, PDP, páginas 2..N) dentro de la MISMA sesión.

La lógica de parseo, formateo de precios, cruce con el diccionario y exportación a Excel
es la misma de tu código original.

Requisitos:
    pip install undetected-chromedriver selenium beautifulsoup4 pandas openpyxl
(undetected-chromedriver es el que ayuda contra el bloqueador; si no está instalado,
 el script cae a Selenium normal con opciones stealth.)
"""

import re
import time
import random
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ----------------------------------------------------------------------------------
# CONFIGURACIÓN
# ----------------------------------------------------------------------------------
HOME_URL = "https://www.suburbia.com.mx/tienda/home"
# URL estable de la categoría Celulares (respaldo si el clic por menú falla).
# El enlace real de la categoría usa este id 'catst...', NO el token codificado viejo.
CELULARES_URL = "https://www.suburbia.com.mx/tienda/celulares/catst53278605"
RUTA_DICCIONARIO = r"C:\Users\c90029859\Documents\diccionario2.xlsx"
HOJA_DICCIONARIO = "Suburbia"
MAX_PAGINAS = 5          # cuántas páginas de resultados recorrer
HEADLESS = False         # OJO: en headless el bloqueador es más agresivo. Déjalo en False.
EXPANDIR_VARIANTES = False  # entrar a cada PDP para variantes color/tamaño (más lento y frágil).
                            # La 'Capacidad' igual se llena desde el diccionario por SKU.
DEBUG_HTML = True        # si una página no trae productos, guarda el HTML para inspección
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")


# ----------------------------------------------------------------------------------
# UTILIDADES DE PARSEO / FORMATO  (idénticas a tu código original)
# ----------------------------------------------------------------------------------
def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""


def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""


def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"


def extraer_codigo_producto(url):
    """CódigoProducto = lo que está ANTES del '?' y DESPUÉS del último '/'.
    Ej: /tienda/pdp/apple-iphone-16e.../SB5012984724?skuid=... -> 'SB5012984724'."""
    m = re.search(r'/([^/?]+)\?', url)
    if m:
        return m.group(1)
    # URL sin query (formato del home): último segmento del path
    m = re.search(r'/tienda/pdp/[^/?]+/([^/?#]+)$', url)
    return m.group(1) if m else ""


def extraer_sku(url):
    """SKU = parámetro skuid de la URL. Ej: ...?skuid=SB5012984733 -> 'SB5012984733'.
    Si no hay skuid (p. ej. enlaces del home), usa el último segmento del path."""
    m = re.search(r'skuid=([^&]+)', url)
    if m:
        return m.group(1)
    m = re.search(r'/tienda/pdp/[^/?]+/([^/?#]+)', url)
    return m.group(1) if m else ""


# Marcas conocidas para distinguir 'Marca' del 'Nombre' en el listado sin depender de clases
MARCAS_CONOCIDAS = {
    "SAMSUNG", "APPLE", "IPHONE", "MOTOROLA", "XIAOMI", "REDMI", "OPPO", "HONOR",
    "ZTE", "NUBIA", "HUAWEI", "REALME", "INFINIX", "NOKIA", "VIVO", "TCL",
    "LG", "ALCATEL", "LANIX", "HISENSE", "POCO", "GHIA",
}

_RE_PRECIO = re.compile(r'\$\s?[\d,]+(?:\.\d{2}|\s\d{2})?')
_RE_DESCUENTO = re.compile(r'-\s?\d{1,3}\s?%')


def _norm_precio(p):
    """Normaliza un precio, reuniendo los centavos si venían en superíndice/aparte.
    '$4,749 29' -> '$4,749.29' ; '$6,999.00' -> '$6,999.00' ; '$2,499' -> '$2,499'."""
    p = p.strip()
    m = re.match(r'\$\s?([\d,]+)(?:\.(\d{2})|\s(\d{2}))?$', p)
    if not m:
        return p.replace(" ", "")
    entero = m.group(1)
    cents = m.group(2) or m.group(3)
    return f"${entero}.{cents}" if cents else f"${entero}"


def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch", "watch", "reloj"]
    return any(p in nombre for p in palabras)


# ----------------------------------------------------------------------------------
# DRIVER STEALTH  (una sola instancia para toda la sesión)
# ----------------------------------------------------------------------------------
def build_driver():
    """Crea un driver lo más 'humano' posible. Prefiere undetected-chromedriver."""
    try:
        import undetected_chromedriver as uc
        options = uc.ChromeOptions()
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--lang=es-MX")
        options.add_argument(f"user-agent={USER_AGENT}")
        driver = uc.Chrome(options=options, headless=HEADLESS)
        print("Driver: undetected-chromedriver")
    except Exception as e:
        print(f"undetected-chromedriver no disponible ({e}). Uso Selenium normal + stealth.")
        options = Options()
        if HEADLESS:
            options.add_argument("--headless=new")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--lang=es-MX")
        options.add_argument(f"user-agent={USER_AGENT}")
        # Ocultar señales de automatización
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        driver = webdriver.Chrome(options=options)
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"},
        )
    driver.set_page_load_timeout(60)
    return driver


def pausa(a=1.2, b=2.8):
    """Espera aleatoria para parecer humano."""
    time.sleep(random.uniform(a, b))


def scroll_pagina(driver, veces=3):
    for _ in range(veces):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        pausa(0.8, 1.6)
    driver.execute_script("window.scrollTo(0, 0);")
    pausa(0.4, 0.8)


def esperar_productos(driver, timeout=30, minimo=4):
    """Espera a que el SPA renderice tarjetas de producto.
    No depende de una clase concreta: cuenta enlaces a /tienda/pdp/.
    Va haciendo scroll para disparar el lazy-load."""
    fin = time.time() + timeout
    while time.time() < fin:
        anclas = driver.find_elements(By.CSS_SELECTOR, "a[href*='/tienda/pdp/']")
        if len(anclas) >= minimo:
            scroll_pagina(driver, 2)  # asegura que carguen todas
            return True
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.2)
    return False


def volcar_html(driver, etiqueta):
    """Guarda el HTML renderizado para poder inspeccionar/ajustar selectores."""
    if not DEBUG_HTML:
        return
    nombre = f"suburbia_debug_{etiqueta}.html"
    try:
        with open(nombre, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"  [debug] HTML guardado en {nombre}")
    except Exception as e:
        print(f"  [debug] no pude guardar HTML: {e}")


def click_robusto(driver, xpaths, descripcion="", timeout=15):
    """Intenta hacer clic en el primer XPath que exista y sea clickeable.
    Prueba clic normal y, si falla, clic por JavaScript."""
    if isinstance(xpaths, str):
        xpaths = [xpaths]
    fin = time.time() + timeout
    while time.time() < fin:
        for xp in xpaths:
            try:
                el = driver.find_element(By.XPATH, xp)
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                pausa(0.4, 0.9)
                try:
                    el.click()
                except Exception:
                    driver.execute_script("arguments[0].click();", el)
                print(f"  clic OK -> {descripcion or xp}")
                return True
            except Exception:
                continue
        time.sleep(0.6)
    print(f"  [!] No pude hacer clic en: {descripcion or xpaths}")
    return False


# ----------------------------------------------------------------------------------
# NAVEGACIÓN COMO EN EL VIDEO
#   home -> Categorías -> Electrónica -> Celulares -> Vendido por: suburbia -> paginar
# ----------------------------------------------------------------------------------
def _href_celulares(driver):
    """Busca en el DOM el enlace REAL de la categoría Celulares (por href, no por texto).
    El enlace correcto apunta a /tienda/celulares/catst..., a diferencia de otros textos
    'Celulares' (breadcrumb, banners) que confundían al selector viejo."""
    selectores = [
        "a[href*='/tienda/celulares/catst']",
        "a[href*='/tienda/celulares/']",
    ]
    for css in selectores:
        try:
            for el in driver.find_elements(By.CSS_SELECTOR, css):
                href = el.get_attribute("href") or ""
                # Nos quedamos SOLO con el listado de categoría, no con PDPs ni el atajo 's='
                if "/tienda/celulares/" in href and "/pdp/" not in href:
                    return href
        except Exception:
            continue
    return None


def abrir_categoria_celulares(driver):
    """Replica el video (home -> Categorías -> Celulares) pero seleccionando Celulares
    de forma DETERMINISTA por su href, para no equivocarse de elemento."""
    print("Abriendo menú Categorías...")
    click_robusto(
        driver,
        ["//button[contains(normalize-space(.),'Categorías')]",
         "//a[contains(normalize-space(.),'Categorías')]",
         "//*[contains(@class,'categor') and (self::button or self::a)]",
         "//*[normalize-space(text())='Categorías']"],
        descripcion="Categorías",
        timeout=8,
    )
    pausa()

    # (Opcional) pasar el mouse por Electrónica para desplegar su sub-lista.
    try:
        electronica = driver.find_element(
            By.XPATH, "//*[self::a or self::li or self::span]"
                      "[normalize-space(text())='Electrónica']")
        ActionChains(driver).move_to_element(electronica).perform()
        pausa(0.6, 1.2)
    except Exception:
        pass

    # Selección determinista de Celulares por href.
    print("Seleccionando Celulares (por href)...")
    href = _href_celulares(driver)
    destino = href or CELULARES_URL
    if href:
        print(f"  Enlace de categoría encontrado: {href}")
    else:
        print(f"  No hallé el enlace en el menú; uso URL estable: {destino}")
    # Navegamos DENTRO de la misma sesión (ya con cookies/token anti-bot del home)
    driver.get(destino)

    # Esperar a que el SPA renderice el listado (independiente de la clase de tarjeta)
    if not esperar_productos(driver, timeout=30):
        print("  [!] No cargaron productos. Guardo HTML para inspección.")
        volcar_html(driver, "categoria")
    print(f"  En categoría Celulares -> {driver.current_url}")


def filtrar_vendido_por_suburbia(driver):
    """Marca el checkbox 'Vendido por: suburbia' en la barra lateral de filtros."""
    print("Aplicando filtro 'Vendido por: suburbia'...")
    ok = click_robusto(
        driver,
        # Distintas formas de alcanzar el checkbox/etiqueta 'suburbia' del facet
        ["//*[contains(@class,'facet') or contains(@class,'filter') or contains(@class,'Vendido')]"
         "//label[contains(translate(., 'SUBRIA', 'subria'),'suburbia')]",
         "//label[contains(translate(., 'SUBRIA', 'subria'),'suburbia')]",
         "//span[contains(translate(., 'SUBRIA', 'subria'),'suburbia')]"
         "/preceding-sibling::input[@type='checkbox']",
         "//*[normalize-space(text())='suburbia' or starts-with(normalize-space(text()),'suburbia')]"],
        descripcion="checkbox 'suburbia'",
    )
    if ok:
        pausa(2.0, 3.5)  # el listado se recarga con el filtro aplicado
        esperar_productos(driver, timeout=20)
    else:
        print("  [!] No se pudo marcar el filtro por clic. Revisa el selector del checkbox "
              "'suburbia' (inspecciona la barra 'Vendido por').")
    print(f"  Filtro aplicado -> {driver.current_url}")
    return driver.current_url


def ir_a_pagina(driver, n, url_base_categoria):
    """Cambia a la página n. Primero intenta hacer clic en el botón de paginación
    (como en el video). Si no, navega por URL DENTRO de la misma sesión (las cookies
    ya están puestas, así que suele pasar el bloqueador)."""
    print(f"Cambiando a página {n}...")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    pausa(0.8, 1.6)
    ok = click_robusto(
        driver,
        [f"//nav//a[normalize-space(text())='{n}']",
         f"//a[normalize-space(text())='{n}' and (contains(@class,'page') or ancestor::*[contains(@class,'pag')])]",
         f"//button[normalize-space(text())='{n}']",
         f"//a[normalize-space(text())='{n}']"],
        descripcion=f"página {n}",
        timeout=8,
    )
    if not ok:
        # Fallback: misma sesión, URL con /page-n
        base = url_base_categoria.split("/page-")[0]
        page_url = base if n == 1 else f"{base}/page-{n}"
        print(f"  Fallback por URL: {page_url}")
        driver.get(page_url)
    if not esperar_productos(driver, timeout=25):
        print("  [!] No cargaron productos en esta página.")
        volcar_html(driver, f"pagina{n}")


# ----------------------------------------------------------------------------------
# PARSEO DEL LISTADO (misma lógica de tu código original)
# ----------------------------------------------------------------------------------
def _contenedor_tarjeta(anchor):
    """Sube desde el enlace /pdp/ hasta el ancestro que representa UNA tarjeta
    (el mayor bloque que sigue conteniendo un solo enlace /pdp/)."""
    nodo = anchor
    contenedor = anchor
    for _ in range(6):
        padre = nodo.parent
        if padre is None:
            break
        # si el padre ya abarca 2+ productos, nos pasamos: nos quedamos con 'nodo'
        if len(padre.select("a[href*='/tienda/pdp/']")) > 1:
            break
        nodo = padre
        contenedor = nodo
    return contenedor


def _extraer_marca_nombre(textos):
    """De la lista de textos de la tarjeta, deduce (marca, nombre).
    Dos pasadas: 1) marca (priorizando marcas conocidas), 2) nombre entre lo restante.
    Corrige nombres cortos en mayúsculas tipo 'G17' que antes se perdían."""
    limpios = []
    for t in textos:
        tt = t.strip()
        if not tt:
            continue
        if _RE_PRECIO.search(tt) or _RE_DESCUENTO.search(tt):
            continue
        if not re.search(r'[A-Za-zÁÉÍÓÚÑáéíóúñ]', tt):  # descarta ratings tipo "(5)"
            continue
        limpios.append(tt)

    # 1) MARCA: primero una marca conocida; si no, un token corto todo-mayúsculas
    marca = ""
    for tt in limpios:
        if tt.upper() in MARCAS_CONOCIDAS:
            marca = tt
            break
    if not marca:
        for tt in limpios:
            if tt == tt.upper() and len(tt) <= 15 and " " not in tt:
                marca = tt
                break

    # 2) NOMBRE: el mejor candidato que NO sea la marca
    candidatos = [tt for tt in limpios if tt.upper() != marca.upper()]
    nombre = ""
    if candidatos:
        clave = ("pulgadas", "amoled", "lcd", "poled", "oled", "pantalla", "gb", "5g")
        pref = [c for c in candidatos if any(k in c.lower() for k in clave)]
        if not pref:
            # los modelos casi siempre traen un número (G17, A17, Reno 12...)
            pref = [c for c in candidatos if re.search(r'\d', c)]
        nombre = max(pref or candidatos, key=len)
    return marca, nombre


def _extraer_colores(contenedor):
    """Lee los swatches de color de forma conservadora para NO inflar filas.
    Solo cuenta elementos que claramente son swatches (data-color o clase con 'color'/
    'swatch'). Si no encuentra, devuelve una sola fila con color vacío."""
    colores = []
    vistos = set()
    swatches = contenedor.select(
        "[data-color], [class*='color' i], [class*='swatch' i], [class*='atom-color' i]"
    )
    for el in swatches:
        nombre = (el.get("title") or el.get("aria-label") or "").strip()
        hexc = (el.get("data-color") or "").strip()
        if nombre and (_RE_PRECIO.search(nombre) or len(nombre) > 25):
            nombre = ""
        clave = (nombre.lower(), hexc.lower())
        if (nombre or hexc) and clave not in vistos:
            vistos.add(clave)
            colores.append((nombre, hexc))
    return colores or [("", "")]


def parsear_listado(driver, fecha_consulta):
    """Extrae de la página actual las filas base + las URLs de PDP a expandir.
    Es INDEPENDIENTE de las clases: se ancla en los enlaces /tienda/pdp/."""
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Un producto por enlace /pdp/ único
    anclas = soup.select("a[href*='/tienda/pdp/']")
    productos = {}
    for a in anclas:
        href = a.get("href", "")
        if "/tienda/pdp/" not in href:
            continue
        url = href if href.startswith("http") else "https://www.suburbia.com.mx" + href
        if url not in productos:
            productos[url] = _contenedor_tarjeta(a)
    print(f"  Productos en esta página: {len(productos)}")

    filas = []
    pdp_para_expandir = []

    for url, contenedor in productos.items():
        textos = list(contenedor.stripped_strings)
        texto_full = " ".join(textos)

        marca, nombre = _extraer_marca_nombre(textos)
        if es_smartwatch(nombre):
            continue

        # IMPORTANTE: quitar la MENSUALIDAD ("Desde $X quincenales/mensuales") antes de
        # extraer precios; si no, ese pago (ej. $336.50) se colaba como Precio_normal.
        texto_precios = re.sub(
            r'(?i)\bdesde\b.*?(?:quincenales|mensuales|semanales|al\s+mes)',
            ' ', texto_full)

        # precios: todos los importes $... en orden de aparición (con centavos reunidos)
        precios = [_norm_precio(p) for p in _RE_PRECIO.findall(texto_precios)]
        es_rango = " - " in texto_precios and len(precios) >= 2 and not _RE_DESCUENTO.search(texto_precios)

        colores = _extraer_colores(contenedor)  # (se conserva por si activas variantes)

        # Lógica de precios equivalente a tu original:
        #   Precio_normal   = precio ORIGINAL (tachado, mayor)
        #   Precio_descuento= precio de OFERTA (rojo, menor)  -> si no hay, "No discount"
        if es_rango:
            precio_normal_final = f"{precios[0]} - {precios[1]}"
            precio_desc_final = "No discount"
            if EXPANDIR_VARIANTES:
                pdp_para_expandir.append({"url": url, "marca": marca, "nombre": nombre})
        elif len(precios) >= 2:
            precio_desc_final = precios[0]      # oferta (rojo)
            precio_normal_final = precios[1]    # original (tachado)
        elif len(precios) == 1:
            precio_normal_final = precios[0]
            precio_desc_final = "No discount"
        else:
            precio_normal_final = ""
            precio_desc_final = "No discount"

        # Una fila por producto (Color vacío en el PLP; se llena en la PDP si expandes).
        filas.append({
            "Marca": marca,
            "Nombre": nombre,
            "Color": "",
            "Capacidad": "",  # se llenará desde el diccionario por SKU
            "CodigoProducto": extraer_codigo_producto(url),
            "SKU": extraer_sku(url),
            "URL": url,
            "FechaConsulta": fecha_consulta,
            "Precio_normal": precio_normal_final,
            "Precio_descuento": precio_desc_final,
        })

    return filas, pdp_para_expandir


# ----------------------------------------------------------------------------------
# EXPANSIÓN DE PDP (variantes color/tamaño) reutilizando la MISMA sesión
# ----------------------------------------------------------------------------------
def expandir_pdp(driver, url, fecha_consulta):
    """Abre la PDP en la misma sesión y recorre las variantes de color/tamaño.
    Devuelve filas listas para el DataFrame (misma lógica que tu original)."""
    filas = []
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
        )
        pausa(1.0, 2.0)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        marca = ""
        nombre = ""
        marca_tag = soup.find("p", class_="brandTitleOffBrand") or soup.find("span", class_="a-product__brand")
        if marca_tag:
            marca = marca_tag.text.strip()
        nombre_tag = soup.find("h1", class_="a-product__information--title") or soup.find("h1")
        if nombre_tag:
            nombre = nombre_tag.text.strip()

        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()

        def leer_variante(soup_var):
            color = codigo = capacidad = precio_normal = precio_desc = ""
            color_tag = soup_var.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                cs = color_tag.find("span", class_="attributeAccordionTextValue")
                color = cs.text.strip() if cs else ""
            codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                cs = codigo_tag.find("span")
                codigo = cs.text.strip() if cs else ""
            for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                if any(k in h3.text for k in ["Tamaño:", "Capacidad:", "Almacenamiento:"]):
                    cs = h3.find("span", class_="attributeAccordionTextValue")
                    capacidad = cs.text.strip() if cs else ""
                    break
            pn = soup_var.find("p", class_="a-product__paragraphRegularPrice")
            pd_ = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
            precio_normal = pn.get_text(strip=True) if pn else ""
            precio_desc = pd_.get_text(strip=True) if pd_ else ""
            if not precio_normal and precio_desc:
                precio_normal, precio_desc = precio_desc, ""
            return color, codigo, capacidad, precio_normal, precio_desc

        def agregar(color, codigo, capacidad, precio_normal, precio_desc):
            clave = (color, capacidad, codigo)
            if clave in combinaciones_unicas:
                return
            combinaciones_unicas.add(clave)
            pn_fmt = formatear_precio(precio_normal) if precio_normal else ""
            pd_fmt = formatear_precio(precio_desc) if precio_desc else "No discount"
            if not pd_fmt:
                pd_fmt = "No discount"
            filas.append({
                "Marca": marca, "Nombre": nombre, "Color": color, "Capacidad": capacidad,
                "CodigoProducto": codigo, "SKU": extraer_sku(url), "URL": url,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": pn_fmt, "Precio_descuento": pd_fmt,
            })

        if color_buttons == [None] and size_buttons == [None]:
            agregar(*leer_variante(soup))
        else:
            for color_btn in color_buttons:
                if color_btn:
                    skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script(
                            "arguments[0].click();",
                            driver.find_element(By.XPATH, f"//a[@data-skuid='{skuid}']"))
                        pausa(1.2, 2.2)
                    except Exception:
                        pass
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                sl = soup_color.find("ul", id="size-list-container")
                sbs = sl.find_all("button", class_="a-btn--actionpdp") if sl else [None]
                for size_btn in sbs:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script(
                                "arguments[0].click();",
                                driver.find_element(By.XPATH, f"//button[@data-skuid='{skuid}']"))
                            pausa(1.2, 2.2)
                        except Exception:
                            pass
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")
                    if soup_var.find("p", class_="a-productInfo_selledByLink"):
                        continue  # vendido por tercero
                    agregar(*leer_variante(soup_var))
    except Exception as e:
        print(f"  [!] Error expandiendo PDP {url}: {e}")
    return filas


# ----------------------------------------------------------------------------------
# CRUCE CON DICCIONARIO + EXPORTACIÓN  (misma lógica de tu código original)
# ----------------------------------------------------------------------------------
def cruzar_diccionario_y_exportar(final_data):
    if not final_data:
        print("No se encontraron productos. Revisa los selectores de navegación.")
        return None

    df = pd.DataFrame(final_data)
    print(f"\n=== EXTRACCIÓN COMPLETADA ===")
    print(f"Total de filas extraídas: {len(df)}")
    print(f"SKUs únicos: {df['SKU'].nunique()}")

    try:
        diccionario = pd.read_excel(RUTA_DICCIONARIO, sheet_name=HOJA_DICCIONARIO)
        print(f"Diccionario cargado con {len(diccionario)} registros")

        # Normalizar nombres de columnas (acepta minúsculas)
        renombres = {"sku": "SKU", "capacidad": "Capacidad", "color": "Color"}
        for viejo, nuevo in renombres.items():
            if viejo in diccionario.columns and nuevo not in diccionario.columns:
                diccionario.rename(columns={viejo: nuevo}, inplace=True)

        if "SKU" in diccionario.columns:
            # Atributos que intentaremos traer del diccionario por SKU
            atributos = [c for c in ("Capacidad", "Color") if c in diccionario.columns]
            print(f"Atributos disponibles en el diccionario: {atributos or 'ninguno'}")

            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip().str.lower()
            df["SKU"] = df["SKU"].astype(str).str.strip().str.lower()
            df = df[df["SKU"] != ""]
            diccionario = diccionario[diccionario["SKU"] != ""]

            if atributos:
                # NO colapsamos por SKU: si el diccionario tiene varias filas para el
                # mismo SKU (una por color), el merge las expande a varias filas.
                # Solo quitamos filas idénticas exactas.
                dic_limpio = diccionario[["SKU"] + atributos].drop_duplicates()
                coincidencias = set(df["SKU"]).intersection(set(dic_limpio["SKU"]))
                print(f"SKUs que coinciden con el diccionario: {len(coincidencias)}")
                n_antes = len(df)

                df = df.merge(dic_limpio, on="SKU", how="left", suffixes=("_orig", "_dict"))
                # Para cada atributo: usar el del diccionario y, si falta, el que ya traía
                for attr in atributos:
                    col_dict = f"{attr}_dict"
                    col_orig = f"{attr}_orig"
                    if col_dict in df.columns and col_orig in df.columns:
                        df[attr] = df[col_dict].fillna(df[col_orig])
                        df.drop(columns=[col_dict, col_orig], inplace=True)
                    elif col_dict in df.columns:  # el df no tenía esa columna
                        df.rename(columns={col_dict: attr}, inplace=True)
                    # limpiar nulos
                    df[attr] = df[attr].fillna("")
                    asignados = (df[attr].astype(str).str.strip() != "").sum()
                    print(f"  {attr} asignada(o) desde diccionario: {asignados}/{len(df)}")
                if len(df) != n_antes:
                    print(f"  Filas por variantes de color: {n_antes} -> {len(df)}")
            else:
                print("Advertencia: el diccionario no tiene columnas 'Capacidad' ni 'Color'.")
        else:
            print("Advertencia: el diccionario no tiene columna 'SKU'.")
    except FileNotFoundError:
        print(f"Diccionario '{RUTA_DICCIONARIO}' no encontrado. Continúo sin capacidades/colores.")
    except Exception as e:
        print(f"Error con el diccionario: {e}. Continúo sin capacidades/colores.")

    df["Porcentaje_descuento"] = df.apply(
        lambda r: calcular_porcentaje_descuento(r["Precio_normal"], r["Precio_descuento"]), axis=1)

    columnas_final = ["Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU",
                      "URL", "FechaConsulta", "Precio_normal", "Precio_descuento",
                      "Porcentaje_descuento"]
    df = df[columnas_final].drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"suburbia_{fecha_hoy}.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"\n=== PROCESO COMPLETADO ===")
    print(f"Datos exportados a: {nombre_archivo}")
    print(f"Total de productos: {len(df)}")
    return df


# ----------------------------------------------------------------------------------
# FLUJO PRINCIPAL
# ----------------------------------------------------------------------------------
def main():
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    final_data = []
    pdp_queue = []
    visitados_pdp = set()

    driver = build_driver()
    try:
        print(f"Abriendo {HOME_URL} ...")
        driver.get(HOME_URL)
        pausa(3.0, 5.0)  # deja que el sitio ponga cookies/token anti-bot

        # 1) Navegar como en el video
        abrir_categoria_celulares(driver)
        url_base_categoria = filtrar_vendido_por_suburbia(driver)

        # 2) Recorrer páginas (paginación) en la MISMA sesión
        for page in range(1, MAX_PAGINAS + 1):
            if page > 1:
                ir_a_pagina(driver, page, url_base_categoria)
            filas, pdp = parsear_listado(driver, fecha_consulta)
            final_data.extend(filas)
            for item in pdp:
                if item["url"] not in visitados_pdp:
                    visitados_pdp.add(item["url"])
                    pdp_queue.append(item)
            pausa(1.5, 3.0)

        # 3) Expandir PDPs con variantes (secuencial, misma sesión)
        if EXPANDIR_VARIANTES:
            print(f"\nExpandiendo {len(pdp_queue)} productos con variantes...")
        else:
            print("\nVariantes de color/capacidad: se toman del diccionario (EXPANDIR_VARIANTES=False).")
        for i, item in enumerate(pdp_queue, 1):
            print(f"  [{i}/{len(pdp_queue)}] {item['url']}")
            final_data.extend(expandir_pdp(driver, item["url"], fecha_consulta))
            pausa(1.5, 3.0)

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    return cruzar_diccionario_y_exportar(final_data)

## Ejecutar

In [6]:
df = main()
df.head(20) if df is not None else None

undetected-chromedriver no disponible (No module named 'undetected_chromedriver'). Uso Selenium normal + stealth.
Abriendo https://www.suburbia.com.mx/tienda/home ...
Abriendo menú Categorías...
  clic OK -> Categorías
Seleccionando Celulares (por href)...
  Enlace de categoría encontrado: https://www.suburbia.com.mx/tienda/celulares/catst53278605
  En categoría Celulares -> https://www.suburbia.com.mx/tienda/celulares/catst53278605
Aplicando filtro 'Vendido por: suburbia'...
  clic OK -> checkbox 'suburbia'
  Filtro aplicado -> https://www.suburbia.com.mx/tienda/celulares/catst53278605
  Productos en esta página: 56
Cambiando a página 2...
  clic OK -> página 2
  Productos en esta página: 56
Cambiando a página 3...
  clic OK -> página 3
  Productos en esta página: 56
Cambiando a página 4...
  clic OK -> página 4
  Productos en esta página: 56
Cambiando a página 5...
  clic OK -> página 5
  Productos en esta página: 56

Expandiendo 0 productos con variantes...

=== EXTRACCIÓN COMPLETAD

,Marca,Nombre,Color,Capacidad,CodigoProducto,SKU,URL,FechaConsulta,Precio_normal,Precio_descuento,Porcentaje_descuento
0,SAMSUNG,Galaxy a07 lcd 6.7 pulgadas desbloqueado,Violeta,128 GB,SB5015807084,sb5015807212,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$2,999","$2,699",10.0%
1,SAMSUNG,Galaxy a07 lcd 6.7 pulgadas desbloqueado,Negro,128 GB,SB5015807084,sb5015807212,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$2,999","$2,699",10.0%
2,SAMSUNG,Galaxy a07 lcd 6.7 pulgadas desbloqueado,Verde obscuro,128 GB,SB5015807084,sb5015807212,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$2,999","$2,699",10.0%
3,APPLE,iPhone 16 OLED 6.1 pulgadas Telcel,NEGRO,128 GB,SB5016211762,sb5016211771,https://www.suburbia.com.mx/tienda/pdp/iphone-...,2026-09-30 10:47:49,"$17,499",No discount,No discount
4,HUAWEI,Nova 13 OLED 6.7 pulgadas,Negro,256 GB,SB5012787881,sb5012787881,https://www.suburbia.com.mx/tienda/pdp/nova-13...,2026-09-30 10:47:49,"$11,999","$5,099",57.5%
5,SAMSUNG,Galaxy A37 Super AMOLED 6.7 pulgadas,Violeta,256 GB,SB5015872901,sb5015872936,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$7,999","$5,999",25.0%
6,SAMSUNG,Galaxy A36 Super AMOLED 6.7 pulgadas,Gris,128 GB,SB5012919094,sb5012928124,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$7,499","$4,999",33.34%
7,SAMSUNG,Galaxy A36 Super AMOLED 6.7 pulgadas,Negro,128 GB,SB5012919094,sb5012928124,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$7,499","$4,999",33.34%
8,SAMSUNG,Galaxy A55 5G Super AMOLED 6.6 pulgadas,Azul,128 GB,SB5009736362,sb5009736362,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$8,999","$6,499",27.78%
9,SAMSUNG,Galaxy A57 Super AMOLED 6.7 pulgadas,Azul obscuro,256 GB,SB5015406420,sb5015406463,https://www.suburbia.com.mx/tienda/pdp/galaxy-...,2026-09-30 10:47:49,"$11,999","$8,999",25.0%
